In [1]:
!pip install kaggle

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"aminetoumohamed","key":"9929731e7d42472e31ff649bbb86a9cf"}'}

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000

Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
 65% 3.36G/5.20G [03:24<01:46, 18.6MB/s]

In [ ]:
!unzip skin-cancer-mnist-ham10000.zip

In [ ]:
import pandas as pd

df_balanced = pd.read_csv("HAM10000_metadata.csv")

print(df_balanced.head())
print(df_balanced.info())

Partie: Analyse des donnes

In [ ]:
print(df_balanced.describe())

In [ ]:
print(df_balanced.isnull().sum())

In [ ]:
import matplotlib.pyplot as plt

counts = df_balanced['dx'].value_counts()

plt.figure(figsize=(8,5))
counts.plot(kind='bar')
plt.title("Distribution des classes")
plt.xlabel("Type de cancer")
plt.ylabel("Nombre d'images")

# ajouter pourcentage
for i, v in enumerate(counts):
    plt.text(i, v + 50, f"{(v/len(df_balanced)*100):.1f}%", ha='center')

plt.show()

In [ ]:
df_balanced['age'].hist()
plt.title("Distribution de l'âge")
plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(8,5))
sns.countplot(x='dx', hue='sex', data=df_balanced)
plt.xticks(rotation=90)
plt.title("Type de cancer selon le sexe")
plt.show()

In [ ]:
sns.countplot(x='localization', data=df_balanced)
plt.xticks(rotation=90)
plt.title("Localisation des lésions")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.kdeplot(data=df_balanced, x='age', hue='dx', fill=True)
plt.title("Distribution de l'âge par type de cancer")
plt.show()

In [ ]:
sns.boxplot(x=df_balanced['age'])
plt.title("Outliers de l'âge")
plt.show()

In [ ]:
sns.boxplot(x='dx', y='age', data=df_balanced)
plt.xticks(rotation=90)
plt.title("Age vs Type de cancer")
plt.show()

partie : normalisation

In [ ]:
# vérifier
print(df_balanced.isnull().sum())

# remplir l'âge par la médiane
df_balanced['age'].fillna(df_balanced['age'].median(), inplace=True)

# remplir le sexe par la valeur la plus fréquente
df_balanced['sex'].fillna(df_balanced['sex'].mode()[0], inplace=True)

In [ ]:
df_balanced.drop_duplicates(inplace=True)

correction des valeurs manquants

In [ ]:
print(df_balanced.isnull().sum())

suppression des doublons

In [ ]:
print(df_balanced.duplicated().sum())

# corrige des erreurs

In [ ]:
df_balanced = pd.read_csv("HAM10000_metadata.csv")

In [ ]:
print(df_balanced['sex'].value_counts(dropna=False))

In [ ]:
df_balanced['sex'].fillna(df_balanced['sex'].mode()[0], inplace=True)

In [ ]:
df_balanced.columns = df_balanced.columns.str.strip()

In [ ]:
df_balanced.describe()

Correction des deseqiulibre

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_balanced['label'] = le.fit_transform(df_balanced['dx'])

In [ ]:
#min_size = df['dx'].value_counts().min()

#df_balanced = df.groupby('dx').apply(
 #   lambda x: x.sample(min_size, random_state=42)
#).reset_index(drop=True)

In [ ]:
import os
import shutil
import pandas as pd

df_balanced = pd.read_csv("/content/HAM10000_metadata.csv")

base_dir = "/content/skin_dataset"
os.makedirs(base_dir, exist_ok=True)

# créer dossiers classes
for label in df_balanced['dx'].unique():
    os.makedirs(os.path.join(base_dir, label), exist_ok=True)

path1 = "/content/HAM10000_images_part_1"
path2 = "/content/HAM10000_images_part_2"

count = 0

for i, row in df_balanced.iterrows():
    img = row['image_id'] + ".jpg"
    label = row['dx']

    src1 = os.path.join(path1, img)
    src2 = os.path.join(path2, img)
    dst = os.path.join(base_dir, label, img)

    if os.path.exists(src1):
        shutil.copy(src1, dst)
        count += 1
    elif os.path.exists(src2):
        shutil.copy(src2, dst)
        count += 1

print("Images copiées:", count)

In [ ]:
import random

base_dir = "/content/skin_dataset"
output_dir = "/content/skin_split"

for split in ['train','val','test']:
    for cls in os.listdir(base_dir):
        os.makedirs(os.path.join(output_dir, split, cls), exist_ok=True)

for cls in os.listdir(base_dir):
    images = os.listdir(os.path.join(base_dir, cls))
    random.shuffle(images)

    n = len(images)
    train_end = int(0.7*n)
    val_end = int(0.85*n)

    for i, img in enumerate(images):
        src = os.path.join(base_dir, cls, img)

        if i < train_end:
            dst = os.path.join(output_dir, 'train', cls, img)
        elif i < val_end:
            dst = os.path.join(output_dir, 'val', cls, img)
        else:
            dst = os.path.join(output_dir, 'test', cls, img)

        shutil.copy(src, dst)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(rescale=1./255)

train_data = datagen.flow_from_directory(
    "/content/skin_split/train",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_data.classes),
    y=train_data.classes
)

class_weights = dict(enumerate(class_weights))
print(class_weights)

In [ ]:
print(df_balanced['dx'].value_counts())

Transformations des donnes

Encodages des variables

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_dx = LabelEncoder()
df_balanced['label'] = le_dx.fit_transform(df_balanced['dx'])

le_sex = LabelEncoder()
df_balanced['sex'] = le_sex.fit_transform(df_balanced['sex'])

le_loc = LabelEncoder()
df_balanced['localization'] = le_loc.fit_transform(df_balanced['localization'])

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_balanced['label'] = le.fit_transform(df_balanced['dx'])

In [ ]:
print(df_balanced[['dx','label']].head())

In [ ]:
print(dict(zip(le.classes_, range(len(le.classes_)))))

Normalisation

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_balanced['age'] = scaler.fit_transform(df_balanced[['age']])

In [ ]:
print(df_balanced['age'].mean())
print(df_balanced['age'].std())

# Feature engineering

creation des variables

In [ ]:
df_balanced = df_balanced.groupby('dx').apply(
    lambda x: x.sample(df_balanced['dx'].value_counts().min(), random_state=42)
).reset_index(drop=True)

In [ ]:
print(df_balanced['sex'].value_counts())

In [ ]:
import pandas as pd

df_balanced['age_group'] = pd.cut(df_balanced['age'], bins=3, labels=[0,1,2])

In [ ]:
df_balanced['sex'].fillna(df_balanced['sex'].mode()[0], inplace=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_balanced['sex'] = le.fit_transform(df_balanced['sex'])

In [ ]:
df_balanced = df_balanced[df_balanced['sex'].isin([0,1])]

In [ ]:
df_balanced['sex'] = df_balanced['sex'].astype(int)

In [ ]:
print(df_balanced['sex'].unique())

In [ ]:
print(df_balanced.shape)
print(df_balanced['sex'].unique())

In [ ]:
df_balanced = df_balanced[df_balanced['sex'].isin([0,1])]

In [ ]:
print(df_balanced.shape)

In [ ]:
df_balanced['sex_age'] = df_balanced['sex'] * df_balanced['age']

import pandas as pd
df_balanced['age_group'] = pd.cut(df_balanced['age'], bins=3, labels=[0,1,2])
df_balanced['age_group'] = df_balanced['age_group'].astype(int)

Interaction des variables

In [ ]:
# Assure-toi que sex est bien 0/1
df_balanced = df_balanced[df_balanced['sex'].isin([0,1])]

# Interaction
df_balanced['sex_age'] = df_balanced['sex'] * df_balanced['age']

In [ ]:
print(df_balanced[['sex','age','sex_age']].head())

In [ ]:
df_balanced['age_group'] = pd.cut(
    df_balanced['age'],
    bins=[0, 30, 60, 100],
    labels=[0, 1, 2]
)

feature engineering

In [ ]:
import pandas as pd
import numpy as np

# copie (important pour éviter bugs)
df_balanced = df_balanced.copy()

# age → numérique
df_balanced['age'] = pd.to_numeric(df_balanced['age'], errors='coerce')

# supprimer valeurs invalides
df_balanced = df_balanced[df_balanced['age'] >= 0]

# supprimer NaN
df_balanced = df_balanced.dropna(subset=['age', 'sex', 'localization'])

In [ ]:
df_balanced['sex_age'] = df_balanced['sex'] * df_balanced['age']

In [ ]:
top_loc = df_balanced['localization'].value_counts().nlargest(5).index

df_balanced['localization'] = df_balanced['localization'].apply(
    lambda x: x if x in top_loc else 'other'
)

# Re-encode the 'localization' column to ensure uniform numerical types for OneHotEncoder
from sklearn.preprocessing import LabelEncoder
le_loc_re = LabelEncoder()
df_balanced['localization'] = le_loc_re.fit_transform(df_balanced['localization'].astype(str))

In [ ]:
print(df_balanced.head())
print(df_balanced.shape)
print(df_balanced.isnull().sum())

selection des variable

In [ ]:
features = [
    'age',
    'sex',
    'localization',
    'age_group',
    'sex_age'
]

X = df_balanced[features]
y = df_balanced['label']

In [ ]:
df_balanced['localization'] = df_balanced['localization'].astype(str)

In [ ]:
df_balanced['age_group'] = df_balanced['age_group'].astype(float)

In [ ]:
print(df_balanced.dtypes)

 Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
num_cols = ['age', 'sex_age']
cat_cols = ['sex', 'localization', 'age_group']

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

In [ ]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [ ]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [ ]:
X_transformed = preprocessor.fit_transform(X)

In [ ]:
print("Avant :", X.shape)
print("Après :", X_transformed.shape)

choix de modele : On choisit le modèle **MobileNet** pour la détection du cancer de la peau car il est à la fois **léger et efficace**.  De plus, MobileNet est souvent **pré-entraîné sur de grandes bases de données d’images**, ce qui permet de faire du *transfer learning* et d’obtenir de bons résultats même avec un nombre limité d’images de lésions cutanées.


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

charder le model

In [ ]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,      # on enlève la dernière couche
    weights='imagenet'      # poids pré-entraînés
)

In [ ]:
#geler le model
base_model.trainable = False

In [ ]:
#ajouter les couches de classification

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)   # réduit dimensions
x = layers.BatchNormalization()(x)

x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)

output = layers.Dense(7, activation='softmax')(x)  # 7 classes

In [ ]:
model = models.Model(
    inputs=base_model.input,
    outputs=output
)

In [ ]:
import os
import shutil

base_dir = "/content/skin_balanced"
os.makedirs(base_dir, exist_ok=True)

# créer dossiers
for label in df_balanced['dx'].unique():
    os.makedirs(os.path.join(base_dir, label), exist_ok=True)

path1 = "/content/HAM10000_images_part_1"
path2 = "/content/HAM10000_images_part_2"

# copier seulement les images du dataset équilibré
for i, row in df_balanced.iterrows():
    img = row['image_id'] + ".jpg"
    label = row['dx']

In [ ]:
print(os.listdir('/content/skin_balanced'))

In [ ]:
import os
import shutil

base_dir = "/content/skin_balanced"
os.makedirs(base_dir, exist_ok=True)

# créer dossiers par classe
for label in df_balanced['dx'].unique():
    os.makedirs(os.path.join(base_dir, label), exist_ok=True)

path1 = "/content/HAM10000_images_part_1"
path2 = "/content/HAM10000_images_part_2"

count = 0

for i, row in df_balanced.iterrows():
    img = row['image_id'] + ".jpg"
    label = row['dx']

    src1 = os.path.join(path1, img)
    src2 = os.path.join(path2, img)
    dst = os.path.join(base_dir, label, img)

    if os.path.exists(src1):
        shutil.copy(src1, dst)
        count += 1
    elif os.path.exists(src2):
        shutil.copy(src2, dst)
        count += 1

print("Images copiées :", count)

In [ ]:
import os, shutil, random

base_dir = "/content/skin_balanced"
output_dir = "/content/skin_split"

for split in ['train', 'val', 'test']:
    for cls in os.listdir(base_dir):
        os.makedirs(os.path.join(output_dir, split, cls), exist_ok=True)

for cls in os.listdir(base_dir):
    images = os.listdir(os.path.join(base_dir, cls))
    random.shuffle(images)

    n = len(images)
    train_end = int(0.7 * n)
    val_end = int(0.85 * n)

    for i, img in enumerate(images):
        src = os.path.join(base_dir, cls, img)

        if i < train_end:
            dst = os.path.join(output_dir, 'train', cls, img)
        elif i < val_end:
            dst = os.path.join(output_dir, 'val', cls, img)
        else:
            dst = os.path.join(output_dir, 'test', cls, img)

        shutil.copy(src, dst)

In [ ]:
#cahrger les donnes
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.3,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

train_data = datagen.flow_from_directory(
    "/content/skin_split/train",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

val_data = datagen.flow_from_directory(
    "/content/skin_split/val",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

In [ ]:
#optimisation authomatiques
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.3, patience=2)
]

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
#entrainment phase1
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=callbacks
)

In [ ]:
#fine_tunning
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

In [ ]:
#recompile
import tensorflow as tf

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
#evaluation sur test
test_data = datagen.flow_from_directory(
    "/content/skin_split/test",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

test_loss, test_acc = model.evaluate(test_data)
print("Test Accuracy:", test_acc)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    zoom_range=0.4,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_gen = ImageDataGenerator(rescale=1./255)

In [ ]:
#utiliser adw
import tensorflow as tf

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=1e-4,
    weight_decay=1e-5
)

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
#Fin_tunning
base_model.trainable = True

for layer in base_model.layers[:-80]:
    layer.trainable = False

In [ ]:
#learning rate
from tensorflow.keras.callbacks import ReduceLROnPlateau

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=2,
    min_lr=1e-6
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
#ameliore
x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.6)(x)

x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.4)(x)

output = tf.keras.layers.Dense(7, activation='softmax')(x)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_data.classes),
    y=train_data.classes
)

class_weights = dict(enumerate(class_weights))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import tensorflow as tf

# =========================
# 1. ÉVALUATION TEST
# =========================
test_loss, test_acc = model.evaluate(test_data)

print("===================================")
print("Test Accuracy :", test_acc)
print("Test Loss :", test_loss)
print("===================================")

# =========================
# 2. PRÉDICTIONS
# =========================
y_pred = model.predict(test_data)
y_pred_classes = np.argmax(y_pred, axis=1)

y_true = test_data.classes

class_names = list(test_data.class_indices.keys())

# =========================
# 3. RAPPORT CLASSIFICATION
# =========================
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred_classes, target_names=class_names))

# =========================
# 4. MATRICE DE CONFUSION
# =========================
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=class_names,
            yticklabels=class_names)

plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion")
plt.show()

# =========================
# 5. SAUVEGARDE MODÈLE
# =========================
model.save("/content/mobilenet_skin_model.h5")

print("\n Modèle sauvegardé dans /content/mobilenet_skin_model.h5")

In [ ]:
# =========================
# INSTALLATION
# =========================
!pip install -q fastapi uvicorn gradio nest-asyncio python-multipart

# =========================
# STOP ANCIENS PROCESSUS
# =========================
!pkill -f uvicorn
!pkill cloudflared

# =========================
# CLOUD FLARED
# =========================
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# =========================
# IMPORTS
# =========================
import tensorflow as tf
import numpy as np
from PIL import Image
from fastapi import FastAPI, UploadFile, File
import nest_asyncio
import uvicorn
from threading import Thread
import gradio as gr
import requests
import time
import subprocess
import re

# =========================
# CHARGER MODÈLE
# =========================
model = tf.keras.models.load_model("/content/mobilenet_skin_model.h5")

class_names = ['nv','mel','bkl','akiec','bcc','df','vasc']

# =========================
# FASTAPI
# =========================
app = FastAPI()

def preprocess_image(image):
    image = image.resize((224,224))
    image = np.array(image) / 255.0
    image = np.expand_dims(image, axis=0)
    return image

@app.post("/predict")
async def predict(file: UploadFile = File(...)):

    image = Image.open(file.file).convert("RGB")

    img = preprocess_image(image)

    pred = model.predict(img)

    pred_class = class_names[np.argmax(pred)]
    confidence = float(np.max(pred))

    return {
        "class": pred_class,
        "confidence": confidence
    }

# =========================
# LANCER FASTAPI
# =========================
nest_asyncio.apply()

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

Thread(target=run).start()

time.sleep(5)

# =========================
# CLOUD FLARE TUNNEL
# =========================
subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=open("log.txt", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(8)

# =========================
# RÉCUPÉRER URL
# =========================
with open("log.txt", "r") as f:
    log = f.read()

url = re.findall(r"https://[-0-9a-z]+\.trycloudflare\.com", log)[0]

print("API URL :", url)

# =========================
# GRADIO
# =========================
API_URL = url + "/predict"

def predict_image(image):

    image = image.convert("RGB")
    image.save("temp.jpg")

    with open("temp.jpg", "rb") as f:

        response = requests.post(
            API_URL,
            files={"file": ("temp.jpg", f, "image/jpeg")}
        )

    result = response.json()

    return f"Classe : {result['class']} | Confiance : {result['confidence']:.2f}"

interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Skin Cancer Detection"
)

interface.launch(share=True)